# 🎯 Train YOLOv8n Face Detection trên WIDER Face Dataset

Notebook này thực hiện:
1. Cài đặt thư viện + Mount Drive
2. Tải model YOLOv8n pretrained
3. Tải và chuẩn bị dataset WIDER Face → format YOLO
4. Huấn luyện model (auto-save checkpoint → Drive mỗi epoch)
5. Đánh giá và xuất model

### ✨ Tính năng:
- **Tự động resume** nếu có checkpoint trên Drive
- **Đổi tài khoản**: chỉ cần share thư mục checkpoint → notebook tự tìm
- Auto-save `best.pt` + `last.pt` → Drive mỗi epoch

> ⚠️ Chạy trên Google Colab với GPU (T4 trở lên).

## 1. Cài đặt & Mount Drive

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted.')

## 2. ⚙️ CẤU HÌNH

In [ ]:
import os
import glob
import shutil

# ============================================================
# 📌 CẤU HÌNH — CHỈNH Ở ĐÂY NẾU CẦN
# ============================================================

# Tên thư mục checkpoint trên Drive
CKPT_FOLDER_NAME = 'yolov8_face_checkpoints'

# Training params
EPOCHS = 100
BATCH_SIZE = 16       # Giảm xuống 8 nếu OOM
IMG_SIZE = 640
LR0 = 0.01
LRF = 0.001
PATIENCE = 20
WORKERS = 4

# ============================================================
# Tự động tìm thư mục checkpoint (My Drive + Shared)
# ============================================================
DRIVE_CKPT_DIR = None

# Các vị trí tìm kiếm trên Drive
search_paths = [
    f'/content/drive/MyDrive/{CKPT_FOLDER_NAME}',               # My Drive
    f'/content/drive/Shareddrives/*/{CKPT_FOLDER_NAME}',        # Shared Drives
    f'/content/drive/MyDrive/**/{CKPT_FOLDER_NAME}',            # Sub-folders
]

# Tìm thư mục checkpoint có sẵn
found_dirs = []
for pattern in search_paths:
    found_dirs.extend(glob.glob(pattern, recursive=True))

# Ưu tiên thư mục có last.pt
for d in found_dirs:
    if os.path.isdir(d) and os.path.exists(os.path.join(d, 'last.pt')):
        DRIVE_CKPT_DIR = d
        print(f'🔍 Tìm thấy checkpoint: {d}')
        break

# Nếu không tìm thấy → tạo mới ở My Drive
if not DRIVE_CKPT_DIR:
    DRIVE_CKPT_DIR = f'/content/drive/MyDrive/{CKPT_FOLDER_NAME}'
    print(f'🆕 Tạo thư mục checkpoint mới: {DRIVE_CKPT_DIR}')

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

TRAIN_PROJECT = '/content/yolov8_face'
TRAIN_NAME = 'wider_face_v1'
YOLO_DIR = '/content/wider_yolo'

print(f'\n📁 Checkpoint Drive: {DRIVE_CKPT_DIR}')
print(f'📐 Epochs={EPOCHS}, Batch={BATCH_SIZE}, ImgSize={IMG_SIZE}')

# Liệt kê file hiện có
existing = os.listdir(DRIVE_CKPT_DIR)
if existing:
    print(f'\n📂 Files trong thư mục checkpoint:')
    for f in sorted(existing):
        size = os.path.getsize(os.path.join(DRIVE_CKPT_DIR, f)) / 1e6
        print(f'   {f} ({size:.1f} MB)')
else:
    print('\n📂 Thư mục checkpoint trống → Train từ đầu.')

## 3. Tải model YOLOv8n pretrained

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("✅ Đã tải model YOLOv8n pretrained.")
print(model.info())

## 4. Tải dataset WIDER Face

In [ ]:
DATASET_DIR = '/content/wider_face'
os.makedirs(DATASET_DIR, exist_ok=True)

URLS = {
    'train': 'https://huggingface.co/datasets/wider_face/resolve/main/data/WIDER_train.zip',
    'val': 'https://huggingface.co/datasets/wider_face/resolve/main/data/WIDER_val.zip',
    'annot': 'https://huggingface.co/datasets/wider_face/resolve/main/data/wider_face_split.zip',
}

if not os.path.exists(f'{DATASET_DIR}/WIDER_train'):
    print('📥 Đang tải WIDER Face Train (~1.5GB)...')
    !wget -q --show-progress -O {DATASET_DIR}/WIDER_train.zip {URLS['train']}
    !unzip -q -o {DATASET_DIR}/WIDER_train.zip -d {DATASET_DIR}
    !rm -f {DATASET_DIR}/WIDER_train.zip
    print('✅ Done.')
else:
    print('✅ WIDER_train đã tồn tại.')

if not os.path.exists(f'{DATASET_DIR}/WIDER_val'):
    print('📥 Đang tải WIDER Face Val (~362MB)...')
    !wget -q --show-progress -O {DATASET_DIR}/WIDER_val.zip {URLS['val']}
    !unzip -q -o {DATASET_DIR}/WIDER_val.zip -d {DATASET_DIR}
    !rm -f {DATASET_DIR}/WIDER_val.zip
    print('✅ Done.')
else:
    print('✅ WIDER_val đã tồn tại.')

annot_found = glob.glob(f'{DATASET_DIR}/**/wider_face_train_bbx_gt.txt', recursive=True)
if not annot_found:
    print('📥 Đang tải Annotations...')
    !wget -q --show-progress -O {DATASET_DIR}/wider_face_split.zip {URLS['annot']}
    !unzip -q -o {DATASET_DIR}/wider_face_split.zip -d {DATASET_DIR}
    !rm -f {DATASET_DIR}/wider_face_split.zip
    print('✅ Done.')
else:
    print('✅ Annotations đã tồn tại.')

In [ ]:
print('📂 Cấu trúc dataset:')
!find {DATASET_DIR} -maxdepth 3 -type d

train_annot_list = glob.glob(f'{DATASET_DIR}/**/wider_face_train_bbx_gt.txt', recursive=True)
val_annot_list = glob.glob(f'{DATASET_DIR}/**/wider_face_val_bbx_gt.txt', recursive=True)
train_img_list = glob.glob(f'{DATASET_DIR}/**/WIDER_train/images', recursive=True)
val_img_list = glob.glob(f'{DATASET_DIR}/**/WIDER_val/images', recursive=True)

TRAIN_ANNOT_PATH = train_annot_list[0] if train_annot_list else None
VAL_ANNOT_PATH = val_annot_list[0] if val_annot_list else None
TRAIN_IMAGE_DIR = train_img_list[0] if train_img_list else None
VAL_IMAGE_DIR = val_img_list[0] if val_img_list else None

print(f'\n🔍 Auto-detected:')
print(f'  Train annot: {TRAIN_ANNOT_PATH}')
print(f'  Val annot:   {VAL_ANNOT_PATH}')
print(f'  Train imgs:  {TRAIN_IMAGE_DIR}')
print(f'  Val imgs:    {VAL_IMAGE_DIR}')

assert TRAIN_ANNOT_PATH, '❌ Không tìm thấy wider_face_train_bbx_gt.txt!'
assert VAL_ANNOT_PATH, '❌ Không tìm thấy wider_face_val_bbx_gt.txt!'
assert TRAIN_IMAGE_DIR, '❌ Không tìm thấy WIDER_train/images!'
assert VAL_IMAGE_DIR, '❌ Không tìm thấy WIDER_val/images!'
print('\n✅ OK!')

## 5. Chuyển đổi WIDER Face → YOLO format

In [ ]:
from PIL import Image

def convert_wider_to_yolo(annotation_path, image_root, output_image_dir, output_label_dir, min_face_size=10):
    """Chuyển đổi WIDER Face annotation sang YOLO format."""
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)
    stats = {'total_images': 0, 'total_faces': 0, 'skipped_faces': 0}
    with open(annotation_path, 'r') as f:
        while True:
            line = f.readline().strip()
            if not line: break
            image_rel_path = line
            image_path = os.path.join(image_root, image_rel_path)
            num_faces = int(f.readline().strip())
            if num_faces == 0:
                f.readline(); continue
            if not os.path.exists(image_path):
                for _ in range(num_faces): f.readline()
                continue
            try:
                img = Image.open(image_path); img_w, img_h = img.size; img.close()
            except:
                for _ in range(num_faces): f.readline()
                continue
            yolo_labels = []
            for _ in range(num_faces):
                bi = f.readline().strip().split()
                x1, y1, w, h = int(bi[0]), int(bi[1]), int(bi[2]), int(bi[3])
                inv = int(bi[7]) if len(bi) > 7 else 0
                if inv == 1 or w < min_face_size or h < min_face_size:
                    stats['skipped_faces'] += 1; continue
                x1, y1 = max(0, x1), max(0, y1)
                w, h = min(w, img_w - x1), min(h, img_h - y1)
                if w <= 0 or h <= 0:
                    stats['skipped_faces'] += 1; continue
                xc = min(max((x1+w/2)/img_w, 0), 1)
                yc = min(max((y1+h/2)/img_h, 0), 1)
                wn = min(max(w/img_w, 0), 1)
                hn = min(max(h/img_h, 0), 1)
                yolo_labels.append(f'0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
                stats['total_faces'] += 1
            if yolo_labels:
                flat = image_rel_path.replace('/', '_').replace('\\', '_')
                base = os.path.splitext(flat)[0]
                dst = os.path.join(output_image_dir, flat)
                if not os.path.exists(dst): shutil.copy2(image_path, dst)
                with open(os.path.join(output_label_dir, base+'.txt'), 'w') as lf:
                    lf.write('\n'.join(yolo_labels)+'\n')
                stats['total_images'] += 1
    return stats

print('🔄 Đang chuyển đổi Train set...')
ts = convert_wider_to_yolo(TRAIN_ANNOT_PATH, TRAIN_IMAGE_DIR,
    f'{YOLO_DIR}/images/train', f'{YOLO_DIR}/labels/train')
print(f"✅ Train: {ts['total_images']} ảnh, {ts['total_faces']} faces")

print('🔄 Đang chuyển đổi Val set...')
vs = convert_wider_to_yolo(VAL_ANNOT_PATH, VAL_IMAGE_DIR,
    f'{YOLO_DIR}/images/val', f'{YOLO_DIR}/labels/val')
print(f"✅ Val: {vs['total_images']} ảnh, {vs['total_faces']} faces")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random, numpy as np

def visualize_yolo_sample(image_dir, label_dir, n=4):
    images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg','.png'))]
    samples = random.sample(images, min(n, len(images)))
    fig, axes = plt.subplots(1, len(samples), figsize=(5*len(samples), 5))
    if len(samples) == 1: axes = [axes]
    for ax, name in zip(axes, samples):
        img = Image.open(os.path.join(image_dir, name))
        w, h = img.size; ax.imshow(np.array(img))
        lbl = os.path.join(label_dir, os.path.splitext(name)[0]+'.txt')
        if os.path.exists(lbl):
            for line in open(lbl):
                p = line.strip().split()
                if len(p)==5:
                    _, xc, yc, bw, bh = map(float, p)
                    ax.add_patch(patches.Rectangle(((xc-bw/2)*w,(yc-bh/2)*h), bw*w, bh*h,
                        linewidth=2, edgecolor='lime', facecolor='none'))
        ax.set_title(name[:25], fontsize=8); ax.axis('off')
    plt.tight_layout(); plt.show()

visualize_yolo_sample(f'{YOLO_DIR}/images/train', f'{YOLO_DIR}/labels/train')

## 6. Tạo dataset YAML

In [ ]:
import yaml

yaml_path = f'{YOLO_DIR}/wider_face.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({'path': YOLO_DIR, 'train': 'images/train', 'val': 'images/val',
               'nc': 1, 'names': ['face']}, f, default_flow_style=False)
print(f"✅ Dataset YAML: {yaml_path}")
print(open(yaml_path).read())

## 7. 🔄 Tự động tìm checkpoint & Resume

Notebook tự tìm `last.pt` trên Drive (My Drive + thư mục được chia sẻ).
- **Có** → resume training tiếp epoch đã dừng
- **Không** → train từ đầu

In [ ]:
resume_checkpoint = None

# Tìm last.pt trên Drive (My Drive + Shared)
last_pt_path = os.path.join(DRIVE_CKPT_DIR, 'last.pt')

if os.path.exists(last_pt_path) and os.path.getsize(last_pt_path) > 1000:
    resume_checkpoint = last_pt_path
    
    # Đọc thông tin checkpoint
    ckpt = torch.load(resume_checkpoint, map_location='cpu')
    saved_epoch = ckpt.get('epoch', -1)
    saved_best = ckpt.get('best_fitness', 0)
    train_args = ckpt.get('train_args', {})
    saved_epochs = train_args.get('epochs', EPOCHS)
    remaining = saved_epochs - (saved_epoch + 1)
    del ckpt
    
    print(f'🔄 TÌM THẤY CHECKPOINT!')
    print(f'   📍 Path: {resume_checkpoint}')
    print(f'   📊 Đã train: {saved_epoch + 1}/{saved_epochs} epochs')
    print(f'   🏆 Best fitness: {saved_best:.4f}')
    print(f'   ⏭️  Còn lại: {remaining} epochs')
    print(f'\n   → Sẽ RESUME từ epoch {saved_epoch + 2}')
else:
    print(f'🆕 Không tìm thấy checkpoint → Train từ đầu.')
    print(f'   (Đã tìm: {DRIVE_CKPT_DIR})')

## 8. Huấn luyện (auto-save → Drive mỗi epoch)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ============================
# Callbacks: Auto-save → Drive
# ============================
best_map50 = 0.0

# Thư mục lưu checkpoint (luôn lưu vào My Drive nếu thư mục shared read-only)
SAVE_CKPT_DIR = f'/content/drive/MyDrive/{CKPT_FOLDER_NAME}'
os.makedirs(SAVE_CKPT_DIR, exist_ok=True)

def on_train_epoch_end(trainer):
    """Mỗi epoch: copy last.pt → Drive."""
    epoch = trainer.epoch + 1
    last_pt = trainer.save_dir / 'weights' / 'last.pt'
    if last_pt.exists():
        shutil.copy2(str(last_pt), os.path.join(SAVE_CKPT_DIR, 'last.pt'))
        print(f'💾 [Epoch {epoch}] last.pt → Drive ✅')
    if epoch % 10 == 0 and last_pt.exists():
        shutil.copy2(str(last_pt), os.path.join(SAVE_CKPT_DIR, f'epoch_{epoch}.pt'))
        print(f'💾 [Epoch {epoch}] snapshot → Drive ✅')

def on_fit_epoch_end(trainer):
    """Sau validation: lưu best model."""
    global best_map50
    epoch = trainer.epoch + 1
    best_pt = trainer.save_dir / 'weights' / 'best.pt'
    current = trainer.metrics.get('metrics/mAP50(B)', 0)
    if best_pt.exists():
        if current > best_map50:
            best_map50 = current
            print(f'🏆 [Epoch {epoch}] NEW BEST! mAP50={best_map50:.4f}')
        shutil.copy2(str(best_pt), os.path.join(SAVE_CKPT_DIR, 'best.pt'))

def on_train_end(trainer):
    """Khi xong: copy final results."""
    w = trainer.save_dir / 'weights'
    for f in ['best.pt', 'last.pt']:
        src = w / f
        if src.exists(): shutil.copy2(str(src), os.path.join(SAVE_CKPT_DIR, f'final_{f}'))
    csv = trainer.save_dir / 'results.csv'
    if csv.exists(): shutil.copy2(str(csv), os.path.join(SAVE_CKPT_DIR, 'results.csv'))
    print(f'\n🎉 Training hoàn tất! Best mAP50: {best_map50:.4f}')
    print(f'📁 Checkpoint: {SAVE_CKPT_DIR}')

# ============================
# Load model: resume hoặc train mới
# ============================
RESUME = False

if resume_checkpoint:
    print(f'🔄 Loading checkpoint...')
    # Tái tạo thư mục training để resume hoạt động
    train_dir = Path(TRAIN_PROJECT) / TRAIN_NAME / 'weights'
    train_dir.mkdir(parents=True, exist_ok=True)
    resume_dst = str(train_dir / 'last.pt')
    if str(resume_checkpoint) != resume_dst:
        shutil.copy2(resume_checkpoint, resume_dst)
    model = YOLO(resume_dst)
    RESUME = True
    print(f'✅ Resume sẵn sàng!')
else:
    model = YOLO('yolov8n.pt')
    print('🆕 Train từ đầu.')

model.add_callback('on_train_epoch_end', on_train_epoch_end)
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
model.add_callback('on_train_end', on_train_end)

In [ ]:
# ============================
# BẮT ĐẦU TRAINING
# ============================
print(f'🚀 {"RESUME" if RESUME else "START"} training!')
print('=' * 50)

try:
    results = model.train(
        data=f'{YOLO_DIR}/wider_face.yaml',
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        lr0=LR0,
        lrf=LRF,
        patience=PATIENCE,
        workers=WORKERS,
        device=0,
        project=TRAIN_PROJECT,
        name=TRAIN_NAME,
        exist_ok=True,
        resume=RESUME,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
        save=True,
        save_period=1,
        plots=True,
        verbose=True,
    )
    print('\n✅ Training hoàn tất!')

except Exception as e:
    if RESUME:
        print(f'\n⚠️ Resume thất bại: {e}')
        print('🔄 FALLBACK: fine-tune từ weights (epoch reset, weights giữ nguyên)\n')
        model = YOLO(resume_checkpoint)
        model.add_callback('on_train_epoch_end', on_train_epoch_end)
        model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
        model.add_callback('on_train_end', on_train_end)
        results = model.train(
            data=f'{YOLO_DIR}/wider_face.yaml',
            epochs=EPOCHS, batch=BATCH_SIZE, imgsz=IMG_SIZE,
            lr0=LR0*0.1, lrf=LRF, patience=PATIENCE, workers=WORKERS,
            device=0, project=TRAIN_PROJECT, name=TRAIN_NAME,
            exist_ok=True, resume=False,
            mosaic=1.0, mixup=0.1, copy_paste=0.1,
            save=True, save_period=1, plots=True, verbose=True,
        )
    else:
        raise e

## 9. Đánh giá model

In [ ]:
best_path = os.path.join(SAVE_CKPT_DIR, 'best.pt')
if not os.path.exists(best_path):
    best_path = f'{TRAIN_PROJECT}/{TRAIN_NAME}/weights/best.pt'

best_model = YOLO(best_path)
metrics = best_model.val(data=f'{YOLO_DIR}/wider_face.yaml', imgsz=640, batch=16, device=0, plots=True)

print("\n📊 Kết quả:")
print(f"  mAP@50:     {metrics.box.map50:.4f}")
print(f"  mAP@50-95:  {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")

In [ ]:
from IPython.display import Image as IPImage, display
result_dir = f'{TRAIN_PROJECT}/{TRAIN_NAME}'
for p in ['results.png','confusion_matrix.png','val_batch0_pred.png','P_curve.png','R_curve.png']:
    fp = os.path.join(result_dir, p)
    if os.path.exists(fp): print(f'\n📈 {p}'); display(IPImage(filename=fp, width=800))

## 10. Test inference

In [ ]:
import matplotlib.pyplot as plt, random
best_model = YOLO(best_path)
val_imgs = [os.path.join(f'{YOLO_DIR}/images/val', f)
            for f in os.listdir(f'{YOLO_DIR}/images/val') if f.endswith(('.jpg','.png'))]
samples = random.sample(val_imgs, min(6, len(val_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(18,12))
for ax, p in zip(axes.flatten(), samples):
    r = best_model.predict(p, conf=0.25, iou=0.45, verbose=False)
    ax.imshow(r[0].plot()[:,:,::-1])
    ax.set_title(f'{len(r[0].boxes)} faces'); ax.axis('off')
plt.suptitle('YOLOv8n Face Detection', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 11. Checkpoint trên Drive

In [ ]:
print(f'📁 {SAVE_CKPT_DIR}')
print('=' * 60)
!ls -lh {SAVE_CKPT_DIR}
print(f'\n🏆 Best mAP50: {best_map50:.4f}')
print(f'\n💡 Đổi tài khoản: Share thư mục "{CKPT_FOLDER_NAME}" → chạy notebook trên TK mới → tự resume!')

## 12. (Tuỳ chọn) Export ONNX

In [ ]:
best_model = YOLO(best_path)
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True, dynamic=False, opset=12)
print(f'✅ ONNX: {onnx_path}')
if onnx_path and os.path.exists(onnx_path):
    shutil.copy2(onnx_path, f'{SAVE_CKPT_DIR}/yolov8n_face_best.onnx')
    print(f'✅ ONNX → {SAVE_CKPT_DIR}/yolov8n_face_best.onnx')

---

## 📝 Tóm tắt

### 💾 Checkpoint tự động:
| File | Cập nhật | Mục đích |
|------|----------|----------|
| `best.pt` | Khi mAP50 ↑ | Model tốt nhất |
| `last.pt` | Mỗi epoch | Resume training |
| `epoch_N.pt` | Mỗi 10 epoch | Snapshot |

### 🔄 Đổi tài khoản:
1. Share thư mục `yolov8_face_checkpoints` trên Drive
2. TK mới: mở notebook → chạy tất cả → **tự tìm & resume**

### Notebook tự động:
```
Có last.pt trên Drive (My Drive hoặc Shared)? → Resume
Không có?                                     → Train từ đầu
Resume fail?                                  → Fallback fine-tune
```